This notebook is intended to show the process of taking our model to production <br> 
using pipelines.<br>
Dataset link : https://www.kaggle.com/datasets/prabinthakur1/heart-failure-prediction-dataset

In [106]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [107]:
heart = pd.read_csv("heart_disease.csv")

In [108]:
X_train,X_test,y_train,y_test = train_test_split(heart.drop(columns=['HeartDisease']),heart['HeartDisease'],test_size=0.2,random_state=42)

In [109]:
list(enumerate(X_train.columns))

[(0, 'Age'),
 (1, 'Sex'),
 (2, 'ChestPainType'),
 (3, 'RestingBP'),
 (4, 'Cholesterol'),
 (5, 'FastingBS'),
 (6, 'RestingECG'),
 (7, 'MaxHR'),
 (8, 'ExerciseAngina'),
 (9, 'Oldpeak'),
 (10, 'ST_Slope'),
 (11, 'Severity')]

In [110]:
heart['RestingECG'].value_counts()

RestingECG
Normal    552
LVH       188
ST        178
Name: count, dtype: int64

# Encoding categorical colummns using column transformer.<br>
# General process:<br>
### encode --> scale --> feature_selection --> implement model  --> predict --> export model 

In [111]:
# note: a column transformer takes list of tuples,and tuples may contain encoders / transformations and columns where those transformations are to be done.
# Encoding Categorical columns 
tfr1 = ColumnTransformer([
    ('encode_ordinal',OrdinalEncoder(categories=[['Down','Flat','Up'],['Low','Medium','High','Critical']],dtype=int,
    handle_unknown='use_encoded_value',unknown_value=-1),[10,11]), # here ordinal cols are encoded.
    ('encode_nominal',OneHotEncoder(drop='first',sparse_output=False,dtype=int,handle_unknown="ignore"),[1,2,6,8]) # here, nominal columns are encoded
],
remainder='passthrough' # the rest are simply pass through to next phase

)

In [112]:
# Scaling the values of all columns as they are now encoded and now are all numeric.
from sklearn.preprocessing import MinMaxScaler
tfr2 = ColumnTransformer([
    ('scaling',MinMaxScaler(),slice(0,14)) # the no of columns after encoding is 14.
])


In [113]:
# now we are doing feature selection.
from sklearn.feature_selection import SelectKBest,chi2
tfr3 = SelectKBest(score_func=chi2,k=11)

In [114]:
# training the model
from sklearn.tree import DecisionTreeClassifier
from sklearn import config_context
tfr4 = DecisionTreeClassifier()

In [115]:
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('tfr1',tfr1),
    ('tfr2',tfr2),
    ('tfr3',tfr3),
    ('tfr4',tfr4)
])

In [116]:
pipe.fit(X_train,y_train)

/Users/prabinthakur/code101/python for ds/ds-env/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/prabinthakur/code101/python for ds/ds-env/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/prabinthakur/code101/python for ds/ds-env/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Pipeline(steps=[('tfr1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encode_ordinal',
                                                  OrdinalEncoder(categories=[['Down',
                                                                              'Flat',
                                                                              'Up'],
                                                                             ['Low',
                                                                              'Medium',
                                                                              'High',
                                                                              'Critical']],
                                                                 dtype=<class 'int'>,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  [10, 11]),
                                                 ('encode_nominal',
                                                  OneHotEncoder(drop='first',
                                                                dtype=<class 'int'>,
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 2, 6, 8])])),
                ('tfr2',
                 ColumnTransformer(transformers=[('scaling', MinMaxScaler(),
                                                  slice(0, 14, None))])),
                ('tfr3',
                 SelectKBest(k=11, score_func=<function chi2 at 0x143d319d0>)),
                ('tfr4', DecisionTreeClassifier())])

In [117]:
y_predict = pipe.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_predict)

0.7771739130434783